In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency

df = pd.read_excel('Research Survey.xlsx')
df['is_bnpl'] = df['Q4_Used_BNPL'].str.contains('Yes')

likert_map = {'Strongly Disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly Agree': 5}
freq_map = {'Very Low': 1, 'Low': 2, 'Moderate': 3, 'High': 4, 'Very High': 5}

df['shop_freq_num'] = df['Q5a_Shopping_Freq'].map(freq_map)
df['spend_num'] = df['Q5b_Typical_Spend'].map(freq_map)
df['price_imp_num'] = df['Q5c_Price_Importance'].map(freq_map)

# Compare Likert items between groups (Q6 items exist for all)
for col in ['Q6a_Compare_Prices','Q6b_Sales_Influence','Q6c_Confident_Without_Trying','Q6d_Comfortable_High_Amounts']:
    df[col+'_num'] = df[col].map(likert_map)

bnpl = df[df['is_bnpl']==True]
non_bnpl = df[df['is_bnpl']==False]

print("=== T-TESTS on Q6 items (both groups) ===")
for col in ['Q6a_Compare_Prices','Q6b_Sales_Influence','Q6c_Confident_Without_Trying','Q6d_Comfortable_High_Amounts']:
    t, p = stats.ttest_ind(bnpl[col+'_num'].dropna(), non_bnpl[col+'_num'].dropna())
    # Fixed the invalid slash in the division here
    d = (bnpl[col+'_num'].mean() - non_bnpl[col+'_num'].mean()) / df[col+'_num'].std()
    print(f"{col}: BNPL={bnpl[col+'_num'].mean():.2f}, Non-BNPL={non_bnpl[col+'_num'].mean():.2f}, t={t:.3f}, p={p:.4f}, d={d:.3f}")

# Chi-square: BNPL use vs shopping frequency category
print("\n=== Chi-square: BNPL vs Shopping Frequency ===")
ct = pd.crosstab(df['is_bnpl'], df['Q5a_Shopping_Freq'])
print(ct)
chi2, p_chi, dof, expected = chi2_contingency(ct)
print(f"Chi2={chi2:.3f}, p={p_chi:.4f}, dof={dof}")

# Chi-square: BNPL use vs typical spend
print("\n=== Chi-square: BNPL vs Typical Spend ===")
ct2 = pd.crosstab(df['is_bnpl'], df['Q5b_Typical_Spend'])
print(ct2)
chi2b, p_chib, dofb, _ = chi2_contingency(ct2)
print(f"Chi2={chi2b:.3f}, p={p_chib:.4f}, dof={dofb}")

# Age by group
print("\n=== Age by group ===")
# Fixed the invalid slashes in the +/- formatting here
print(f"BNPL: {bnpl['Q1_Age'].mean():.2f} +/- {bnpl['Q1_Age'].std():.2f}")
print(f"Non-BNPL: {non_bnpl['Q1_Age'].mean():.2f} +/- {non_bnpl['Q1_Age'].std():.2f}")
t_age, p_age = stats.ttest_ind(bnpl['Q1_Age'], non_bnpl['Q1_Age'])
print(f"t={t_age:.3f}, p={p_age:.4f}")

# Recommend scale distribution
print("\n=== Recommend scale distribution ===")
print(bnpl['Q10_Recommend_Scale_1_10'].value_counts().sort_index())

# Negative experience
print("\n=== Negative experiences ===")
print(bnpl['Q11_Negative_Experience'].value_counts().head(10))

# Non-BNPL reasons
print("\n=== Non-BNPL reasons ===")
print(non_bnpl['Q12_Non_BNPL_Reasons'].value_counts().head(10))

# Likely to try (non-BNPL group)
print("\n=== Likely to try (non-BNPL) ===")
print(non_bnpl['Q13_Likely_to_Try_0_Interest'].describe())

=== T-TESTS on Q6 items (both groups) ===
Q6a_Compare_Prices: BNPL=2.84, Non-BNPL=2.96, t=-0.674, p=0.5011, d=-0.085
Q6b_Sales_Influence: BNPL=3.06, Non-BNPL=3.02, t=0.226, p=0.8212, d=0.029
Q6c_Confident_Without_Trying: BNPL=3.07, Non-BNPL=3.33, t=-1.446, p=0.1495, d=-0.182
Q6d_Comfortable_High_Amounts: BNPL=3.02, Non-BNPL=3.15, t=-0.722, p=0.4709, d=-0.091

=== Chi-square: BNPL vs Shopping Frequency ===
Q5a_Shopping_Freq  High  Low  Moderate  Very High  Very Low
is_bnpl                                                    
False                21   23        28         24        29
True                 28   27        23         25        22
Chi2=2.791, p=0.5933, dof=4

=== Chi-square: BNPL vs Typical Spend ===
Q5b_Typical_Spend  High  Low  Moderate  Very High  Very Low
is_bnpl                                                    
False                30   21        25         29        20
True                 28   25        21         24        27
Chi2=2.279, p=0.6846, dof=4

=== Age by 